![Machine Learning Practical Midterm Exam — Instructor: Masoud Ahangary](assets/banner_exam_cover.png)


# Machine Learning — Practical Midterm Exam

**Total: 100 points**

This notebook is the **practical/programming** part of the midterm. No written conceptual explanations are required.

The notebook is designed as a guided programming workflow. Short notes between tasks explain what stage of the modeling process you are in and why the next programming step is needed.

## Rules
- Complete only the sections marked `YOUR CODE HERE`.
- Do not rename the required functions or change their parameters.
- Do not edit cells marked `DO NOT EDIT`.
- Run the public test after each task. A correct implementation should print `All tests passed!`.
- Public tests help you detect implementation errors, but your code must still be general and readable.
- Plotting code is provided by the instructor. **matplotlib syntax is not part of this exam.**
- Before submission, use **Restart Kernel and Run All** and make sure the notebook runs from top to bottom.

---

![Part A — Regression Challenge: Housing Price Prediction](assets/banner_part_a_regression.png)

# Part A — Regression Challenge (50 points)

In this part, you will first build the main components of a Linear Regression trainer using NumPy. You will then use the same regression problem to work with scaling, scikit-learn Linear Regression, Polynomial Regression, Pipeline, and K-Fold Cross-Validation.


## Complete Machine Learning Workflow

This exam follows a realistic machine learning pipeline.

Regression:

Dataset
→ Train/Test Split
→ Raw Model Training *(on training set)*
→ Feature Scaling *(fit on training set only)*
→ Scaled Training
→ Test-Set Evaluation
→ Cross Validation *(on training set)*

**Important:**
- Scaling is performed **after splitting** to prevent data leakage.
- The scaler learns statistics only from the training set.
- The held-out test set is used for final evaluation, not for training or model selection.


## Environment Setup

Keep `requirements.txt`, `public_tests.py`, `exam_utils.py`, and the two CSV files in the **same folder** as this notebook.

Run the next cell once before starting. It installs the packages required by the exam. If your environment asks for a kernel restart after installation, restart it and continue from the top.

In [ ]:
# DO NOT EDIT — Environment setup
%pip install -q --disable-pip-version-check -r requirements.txt
print("Environment setup completed.")

In [ ]:
# DO NOT EDIT
from __future__ import annotations

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error

from public_tests import *
from exam_utils import (
    show_feature_ranges,
    plot_regression_dataset_overview,
    plot_learning_rate_histories,
    plot_model_selection_scores,
    plot_classification_dataset_overview,
)

np.set_printoptions(suppress=True, precision=4)


In [ ]:
# DO NOT EDIT
reg_df = pd.read_csv("regression_exam.csv")
reg_df.head()

## Regression Dataset and Objective

The file `regression_exam.csv` contains information about **180 residential properties**. Each row represents one property.

| Column | Description |
|---|---|
| `area_m2` | Property area in square meters |
| `rooms` | Number of rooms |
| `building_age` | Building age in years |
| `distance_to_center_km` | Distance from the city center in kilometers |
| `price` | Property price — **target variable** |

### Overall objective

We want to build regression models that predict `price` from property features.

We will not jump directly to a library model. First, you will construct the important pieces of Gradient Descent yourself. The notebook will then gradually move from a simple **raw, single-feature** problem to a **scaled, multiple-feature** problem and finally to scikit-learn model-selection workflows.

## A0 — Prepare the Regression Dataset (3 points)

Before building any model, create a proper machine learning workflow.

Complete `prepare_regression_data`.

**Requirements**
1. Separate features `X` from the target `y` (`price`).
2. Split into training and testing sets with:
   - `test_size=0.2`
   - `random_state=42`
3. Return:

```python
X_train, X_test, y_train, y_test  # NumPy arrays
```

The test set represents unseen data. It must not influence any training decision.

**Important workflow rule:**

`Train/Test Split → Preprocessing → Training → Test Evaluation`

Scaling will be introduced later, and it must be learned only from the training data.


In [ ]:
def prepare_regression_data(
    df: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Split regression features and target into train/test sets.

    Parameters
    ----------
    df : DataFrame
        Full regression dataset including the `price` column.

    Returns
    -------
    X_train, X_test, y_train, y_test : ndarrays
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_A0(prepare_regression_data, reg_df)


In [ ]:
# DO NOT EDIT — Dataset overview
plot_regression_dataset_overview(reg_df)

### First training setup: one raw feature, no scaling

We begin with only `area_m2` as the input feature and `price` as the target.

**Important:** at this stage we use the **training set only**. Feature values are used exactly as they appear after the split. **No scaling is applied yet.**

This first setup keeps the problem simple so that the focus is on implementing the training process itself. After the Gradient Descent workflow is complete, we will introduce multiple features and scaling — still fitted only on the training data.


In [ ]:
# DO NOT EDIT — Build train/test arrays from A0, then single-feature training views
X_train, X_test, y_train, y_test = prepare_regression_data(reg_df)

# area_m2 is the first feature column
X_train_single = X_train[:, [0]]
X_test_single = X_test[:, [0]]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("X_train_single shape:", X_train_single.shape)
print("Scaling applied: No")


### Stage 1 — Build the Gradient Descent Components

The raw single-feature **training** data is ready. We will now build the training process as small reusable functions rather than writing one large block of code.

The sequence is:
`prediction → loss → gradients → parameter update → training loop`

The first building block is the prediction function.


## A1 — Linear Prediction Function (4 points)

Complete `linear_predict`.

The function receives:
- `X`: shape `(m, n)`
- `w`: shape `(n,)`
- `b`: scalar

Return:
```python
y_pred  # ndarray of shape (m,)
```

**Hint:** Matrix multiplication with `@` can be useful.


In [ ]:
def linear_predict(X: np.ndarray, w: np.ndarray, b: float) -> np.ndarray:
    """Compute linear predictions for each row of X.

    Parameters
    ----------
    X : ndarray of shape (m, n)
    w : ndarray of shape (n,)
    b : float

    Returns
    -------
    y_pred : ndarray of shape (m,)
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_A1(linear_predict)

## A2 — Mean Squared Error (4 points)

Complete `compute_mse` without using `sklearn.metrics`.

**Plan**
1. Compute the errors.
2. Square the errors.
3. Return their mean.

Return:
```python
mse  # scalar float
```


In [ ]:
def compute_mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute mean squared error.

    Parameters
    ----------
    y_true : ndarray of shape (m,)
    y_pred : ndarray of shape (m,)

    Returns
    -------
    mse : float
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_A2(compute_mse)

## A3 — Compute the Gradients (7 points)

Complete `compute_gradients` for Linear Regression with MSE.

**Required steps**
1. Compute predictions using `linear_predict`.
2. Compute `error = y_pred - y`.
3. Compute `dw`.
4. Compute `db`.

`dw` must have the same shape as `w`, and `db` must be a scalar.

Return:
```python
dw, db
```

**Hint:** `X.T @ error` combines the contribution of all samples for all features.


In [ ]:
def compute_gradients(
    X: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    b: float,
) -> tuple[np.ndarray, float]:
    """Compute MSE gradients with respect to w and b.

    Parameters
    ----------
    X : ndarray of shape (m, n)
    y : ndarray of shape (m,)
    w : ndarray of shape (n,)
    b : float

    Returns
    -------
    dw : ndarray of shape (n,)
    db : float
    """
    m = len(y)

    # YOUR CODE HERE
    # y_pred = ...
    # error = ...
    # dw = ...
    # db = ...

    pass


In [ ]:
# DO NOT EDIT
test_A3(compute_gradients)

## A4 — Update the Parameters (4 points)

Implement one Gradient Descent update step.

Return the updated parameters:
```python
w, b
```

**Hint:** Parameters move in the opposite direction of the gradient.


In [ ]:
def update_parameters(
    w: np.ndarray,
    b: float,
    dw: np.ndarray,
    db: float,
    learning_rate: float,
) -> tuple[np.ndarray, float]:
    """Apply one gradient-descent parameter update.

    Parameters
    ----------
    w, dw : ndarray of shape (n,)
    b, db : float
    learning_rate : float

    Returns
    -------
    w, b : updated parameters
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_A4(update_parameters)

## A5 — Build the Gradient Descent Trainer (8 points)

Now combine the previous functions.

Your function must:
- initialize `w` with zeros,
- initialize `b = 0.0`,
- repeat Gradient Descent for `num_iterations`,
- store the MSE after every update.

Return:
```python
w, b, cost_history  # cost_history has length num_iterations
```

**Hint:** The number of model weights is `X.shape[1]`.

This task intentionally gives less code scaffolding than the previous tasks.


In [ ]:
def gradient_descent(
    X: np.ndarray,
    y: np.ndarray,
    learning_rate: float,
    num_iterations: int,
) -> tuple[np.ndarray, float, np.ndarray]:
    """Train linear regression with batch gradient descent.

    Parameters
    ----------
    X : ndarray of shape (m, n)
    y : ndarray of shape (m,)
    learning_rate : float
    num_iterations : int

    Returns
    -------
    w : ndarray of shape (n,)
    b : float
    cost_history : ndarray of shape (num_iterations,)
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_A5(gradient_descent)

### Stage 2 — Train on Raw Data with Different Learning Rates

Your Gradient Descent trainer is now complete.

We will **still use the original, unscaled `area_m2` training values** for this experiment. The only thing that changes between runs is the learning rate.

This keeps the experiment controlled: the training data stays the same while the training step size changes. Your task is to generate the cost history for each learning rate; the visualization is handled by a provided utility function.


## A6 — Learning-Rate Experiment (5 points)

Write a function that trains the model once for every learning rate and stores the resulting cost histories.

Return a dictionary in this form:

```python
{
    learning_rate_1: cost_history_1,
    learning_rate_2: cost_history_2,
    ...
}
```

Use your own `gradient_descent` function inside this task.


In [ ]:
def run_learning_rate_experiment(
    X: np.ndarray,
    y: np.ndarray,
    learning_rates: list[float] | np.ndarray,
    num_iterations: int,
) -> dict[float, np.ndarray]:
    """Train once per learning rate and collect cost histories.

    Parameters
    ----------
    X : ndarray of shape (m, n)
    y : ndarray of shape (m,)
    learning_rates : sequence of float
    num_iterations : int

    Returns
    -------
    histories : dict
        Maps each learning rate to its cost_history array.
    """
    histories = {}

    # YOUR CODE HERE

    return histories


In [ ]:
# DO NOT EDIT
test_A6(run_learning_rate_experiment)

The next cell runs your function on the **raw single-feature training data** and calls the provided plotting utility. You do not need to write any matplotlib code.


In [ ]:
# DO NOT EDIT — Run the learning-rate experiment on raw, unscaled training data
learning_rates = [1e-7, 2e-5, 5e-5]
histories = run_learning_rate_experiment(
    X_train_single,
    y_train,
    learning_rates=learning_rates,
    num_iterations=250
)

plot_learning_rate_histories(histories, title="Learning Rate Comparison — Raw area_m2 (train)")


### Stage 3 — Move from One Raw Feature to Multiple Scaled Features

Up to this point, Gradient Descent used only one feature (`area_m2`) and **no feature scaling**, trained on the training set.

We will now use all four training features together:
- `area_m2`
- `rooms`
- `building_age`
- `distance_to_center_km`

These columns use different units and numerical ranges. Before training our Gradient Descent implementation with them together, we will create scaled versions of the feature matrix.

You will implement both:
- `StandardScaler`
- `MinMaxScaler`

**Critical:** call your scaler function on the **training features only**. The next integration cell will then transform the test features with the fitted scaler. Never fit a scaler on the full dataset.


In [ ]:
# DO NOT EDIT — Inspect the raw feature ranges before scaling
_regression_features = ["area_m2", "rooms", "building_age", "distance_to_center_km"]
show_feature_ranges(reg_df, _regression_features, title="Raw regression feature ranges")

## Feature Scaling Workflow

Scaling is part of the machine learning pipeline.

The scaler must learn statistics only from training data:

1. Fit scaler on `X_train`
2. Transform `X_train`
3. Transform `X_test` with the **same** fitted scaler

Never fit the scaler using the complete dataset because that leaks information from the test set.

In `A7` you implement a helper that fits both scalers on whatever matrix you pass in. In the exam workflow you must pass **`X_train` only**, then use the returned scaler objects to transform `X_test`.


## A7 — StandardScaler and MinMaxScaler (5 points)

Complete `scale_features`.

It must:
1. create a `StandardScaler`,
2. create a `MinMaxScaler`,
3. fit and transform `X` with both scalers.

Return:
```python
X_standard, X_minmax, standard_scaler, minmax_scaler
```

In the workflow below, this function is called on **`X_train` only**. The fitted `standard_scaler` is then used to transform `X_test`.


In [ ]:
def scale_features(
    X: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, StandardScaler, MinMaxScaler]:
    """Fit StandardScaler and MinMaxScaler on X and transform it.

    Parameters
    ----------
    X : ndarray of shape (m, n)
        In this exam workflow, pass training features only.

    Returns
    -------
    X_standard : ndarray of shape (m, n)
    X_minmax : ndarray of shape (m, n)
    standard_scaler : fitted StandardScaler
    minmax_scaler : fitted MinMaxScaler
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_A7(scale_features)

In [ ]:
# DO NOT EDIT — Scale training features, transform test, then train Gradient Descent
X_train_standard, X_train_minmax, standard_scaler, minmax_scaler = scale_features(X_train)
X_test_standard = standard_scaler.transform(X_test)

w_multi, b_multi, history_multi = gradient_descent(
    X_train_standard,
    y_train,
    learning_rate=0.05,
    num_iterations=1000
)

y_pred_train = linear_predict(X_train_standard, w_multi, b_multi)
y_pred_test = linear_predict(X_test_standard, w_multi, b_multi)
print("Training input: StandardScaler output (fit on train only)")
print("Train MSE:", compute_mse(y_train, y_pred_train))
print("Test MSE:", compute_mse(y_test, y_pred_test))


### Stage 4 — Use scikit-learn for Linear Regression

You have now trained Linear Regression with your own Gradient Descent code, including a scaled multiple-feature version fitted on the training set.

We will next use scikit-learn's `LinearRegression` to solve the same regression problem with the standard library workflow:
`create model → fit → predict → evaluate`

In this task we use the **unscaled** feature matrices. Fit on the training set; the integration cell also reports test-set MSE.


## A8 — Linear Regression with scikit-learn (3 points)

Complete the function below.

It must:
- create `LinearRegression`,
- fit the model,
- make predictions on `X`,
- compute MSE.

Return:
```python
model, mse
```


In [ ]:
def fit_sklearn_linear_regression(
    X: np.ndarray,
    y: np.ndarray,
) -> tuple[LinearRegression, float]:
    """Fit scikit-learn LinearRegression and report in-sample MSE.

    Parameters
    ----------
    X : ndarray of shape (m, n)
    y : ndarray of shape (m,)

    Returns
    -------
    model : fitted LinearRegression
    mse : float
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_A8(fit_sklearn_linear_regression)

In [ ]:
# DO NOT EDIT
sklearn_model, sklearn_train_mse = fit_sklearn_linear_regression(X_train, y_train)
sklearn_test_mse = mean_squared_error(y_test, sklearn_model.predict(X_test))
print("scikit-learn Train MSE:", sklearn_train_mse)
print("scikit-learn Test MSE:", sklearn_test_mse)


### Stage 5 — Move from Linear to Polynomial Regression

So far, the models have used a linear representation of the input features.

For the final regression stage, we return to the single feature `area_m2` and create polynomial versions of it. Instead of manually choosing one degree, you will compare several candidate degrees using K-Fold Cross-Validation **on the training set** (the held-out test set stays unused during model selection).

Each candidate must be built as a scikit-learn `Pipeline`:
`PolynomialFeatures → LinearRegression`

The workflow is:
`candidate degree → Pipeline → K-Fold CV → mean score → best degree`

Plotting the CV scores is handled by a provided utility function.


## A9 — Polynomial Regression, Pipeline, and K-Fold CV (7 points)

Complete `select_best_polynomial_degree`.

For each degree:
- create a `Pipeline` containing `PolynomialFeatures` and `LinearRegression`,
- run K-Fold Cross-Validation,
- compute the mean R² score,
- store one mean score per degree.

Return:
```python
best_degree, mean_scores
```

**Requirements**
- `KFold(n_splits=n_splits, shuffle=True, random_state=42)`
- `scoring="r2"`
- `include_bias=False`
- choose the best degree using the largest mean CV score.

**Hint:** `np.argmax()` may be useful.


In [ ]:
def select_best_polynomial_degree(
    X: np.ndarray,
    y: np.ndarray,
    degrees: np.ndarray,
    n_splits: int = 5,
) -> tuple[int, np.ndarray]:
    """Select the polynomial degree with the best mean CV R².

    Parameters
    ----------
    X : ndarray of shape (m, n)
    y : ndarray of shape (m,)
    degrees : ndarray of candidate degrees
    n_splits : int

    Returns
    -------
    best_degree : int
    mean_scores : ndarray of shape (len(degrees),)
    """
    mean_scores = []

    # YOUR CODE HERE

    return best_degree, np.array(mean_scores)


In [ ]:
# DO NOT EDIT
test_A9(select_best_polynomial_degree)

In [ ]:
# DO NOT EDIT — Run polynomial degree selection on the training set
degrees = np.arange(1, 7)

best_degree, degree_scores = select_best_polynomial_degree(
    X_train_single,
    y_train,
    degrees,
    n_splits=5
)

print("Best degree:", best_degree)

plot_model_selection_scores(
    degrees,
    degree_scores,
    xlabel="Polynomial Degree",
    ylabel="Mean CV R2",
    title="Polynomial Degree Selection (train CV)"
)


![Part B — KNN Classifier: Customer Churn Classification](assets/banner_part_b_knn.png)


---
# Part B — Build Your Own KNN Classifier (50 points)

In this challenge, you will implement the core KNN prediction process using Python and NumPy.

You are **not expected to memorize a complete KNN implementation**.  
Each task gives you the required algorithmic step. Your job is to translate that plan into working code.

The hints become shorter as the challenge progresses.

In [ ]:
# DO NOT EDIT
cls_df = pd.read_csv("classification_exam.csv")
cls_df.head()

## Classification Dataset and Objective

The file `classification_exam.csv` contains customer information for a **binary churn classification** problem.

| Column | Description |
|---|---|
| `age` | Customer age |
| `monthly_spending` | Average monthly spending |
| `support_calls` | Number of support calls |
| `months_as_customer` | Customer relationship length in months |
| `churn` | Target class: `0` = no churn, `1` = churn |

### Overall objective

We want to predict whether each customer belongs to class `0` or class `1` using K-Nearest Neighbors.

Unlike the regression section, KNN is a **distance-based** method. Because distances are directly affected by feature magnitude, the first step will be to prepare **scaled** training and test feature arrays before implementing KNN from scratch.

In [ ]:
# DO NOT EDIT — Dataset overview
plot_classification_dataset_overview(cls_df)

_classification_features = ["age", "monthly_spending", "support_calls", "months_as_customer"]
show_feature_ranges(cls_df, _classification_features, title="Raw classification feature ranges")

### Stage 1 — Scale the Data Before Computing KNN Distances

The raw classification features have different numerical ranges. Since our KNN implementation will compare samples using Euclidean distance, we do **not** want to compute those distances on the raw feature matrix.

We will therefore:
1. split the raw data into training and test sets,
2. fit `StandardScaler` on the training features only,
3. transform both training and test features,
4. use the resulting scaled arrays in every from-scratch KNN task that follows.

This is different from the beginning of Part A: the first regression experiment intentionally used raw single-feature data, while the KNN workflow begins with scaling because distance is central to the algorithm.

## B0 — Prepare the Data for KNN (4 points)

Complete `prepare_knn_data`.

Requirements:
- target column: `churn`
- `test_size=0.25`
- `random_state=42`
- `stratify=y`
- fit `StandardScaler` **only on the training features**
- transform both train and test features

Return:
```python
X_train_scaled, X_test_scaled, y_train, y_test, scaler
```


In [ ]:
def prepare_knn_data(
    df: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, StandardScaler]:
    """Split, stratify, and scale features for KNN.

    Parameters
    ----------
    df : DataFrame
        Classification dataset including the `churn` column.

    Returns
    -------
    X_train_scaled, X_test_scaled, y_train, y_test, scaler
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_B0(prepare_knn_data, cls_df)

In [ ]:
# DO NOT EDIT
X_train_scaled, X_test_scaled, y_train, y_test, scaler = prepare_knn_data(cls_df)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

### Stage 2 — Build KNN from Small Reusable Functions

The train/test data is now scaled and ready for a distance-based model.

Instead of writing an entire KNN classifier at once, you will build it step by step:
`distance → nearest indices → neighbor labels → class probabilities → prediction`

Each new function should reuse the previous building blocks where appropriate.

## B1 — Euclidean Distance (4 points)

Complete the function that computes the Euclidean distance between two samples.

**Plan**
1. Subtract the vectors.
2. Square the differences.
3. Sum them.
4. Take the square root.

Return:
```python
distance  # scalar float
```

Use only Python/NumPy.


In [ ]:
def euclidean_distance(x1: np.ndarray, x2: np.ndarray) -> float:
    """Compute Euclidean distance between two vectors.

    Parameters
    ----------
    x1, x2 : ndarray of shape (n,)

    Returns
    -------
    distance : float
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_B1(euclidean_distance)

## B2 — Distance to Every Training Sample (4 points)

For one `x_query`, compute its distance to every row in `X_train`.

Return:
```python
distances  # ndarray of shape (m,)
```

**Hint**
- A loop is allowed.
- Reuse `euclidean_distance`.


In [ ]:
def compute_distances(X_train: np.ndarray, x_query: np.ndarray) -> np.ndarray:
    """Compute distance from one query to every training row.

    Parameters
    ----------
    X_train : ndarray of shape (m, n)
    x_query : ndarray of shape (n,)

    Returns
    -------
    distances : ndarray of shape (m,)
    """
    distances = []

    # YOUR CODE HERE

    return np.array(distances)


In [ ]:
# DO NOT EDIT
test_B2(compute_distances)

## B3 — Find the k Nearest Samples (5 points)

Return the indices of the `k` nearest training samples.

Return:
```python
neighbor_indices  # ndarray of shape (k,)
```

**Steps**
1. Compute all distances.
2. Sort the sample indices by distance.
3. Keep the first `k` indices.

**Hint:** `np.argsort()` returns indices in sorted order.


In [ ]:
def get_k_nearest_indices(
    X_train: np.ndarray,
    x_query: np.ndarray,
    k: int,
) -> np.ndarray:
    """Return indices of the k nearest training samples.

    Parameters
    ----------
    X_train : ndarray of shape (m, n)
    x_query : ndarray of shape (n,)
    k : int

    Returns
    -------
    neighbor_indices : ndarray of shape (k,)
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_B3(get_k_nearest_indices)

## B4 — Get the Neighbor Labels (3 points)

Use the neighbor indices to select the corresponding values from `y_train`.

Return:
```python
neighbor_labels  # ndarray of shape (k,)
```

Reuse `get_k_nearest_indices`.


In [ ]:
def get_neighbor_labels(
    X_train: np.ndarray,
    y_train: np.ndarray,
    x_query: np.ndarray,
    k: int,
) -> np.ndarray:
    """Return labels of the k nearest neighbors.

    Parameters
    ----------
    X_train : ndarray of shape (m, n)
    y_train : ndarray of shape (m,)
    x_query : ndarray of shape (n,)
    k : int

    Returns
    -------
    neighbor_labels : ndarray of shape (k,)
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_B4(get_neighbor_labels)

## B5 — Estimate Class Probabilities (5 points)

This is a binary classification problem with labels `0` and `1`.

Estimate each class probability using the fraction of neighbors belonging to that class.

Example:

```python
neighbor_labels = [1, 0, 1, 1, 0]
```

gives:
- `P(class=0) = 2/5`
- `P(class=1) = 3/5`

Return:
```python
prob_0, prob_1
```

**Hint:** `neighbor_labels == 1` produces a Boolean array.


In [ ]:
def compute_class_probabilities(
    neighbor_labels: np.ndarray,
) -> tuple[float, float]:
    """Estimate binary class probabilities from neighbor labels.

    Parameters
    ----------
    neighbor_labels : ndarray of shape (k,)
        Labels are 0 or 1.

    Returns
    -------
    prob_0 : float
    prob_1 : float
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_B5(compute_class_probabilities)

### Stage 3 — Combine the KNN Building Blocks

You have implemented the individual pieces needed for KNN. The next task combines them into a single-sample prediction function.

The input samples here are still the **scaled arrays created in B0**. No new scaling step is needed inside the from-scratch KNN functions.

## B6 — Predict One Sample (7 points)

This is the first integration task.

Reuse your previous functions to:
- get the neighbor labels,
- compute `prob_1`,
- predict class `1` when `prob_1 >= 0.5`, otherwise class `0`.

Return:
```python
prediction, prob_1
```

No additional implementation steps are provided.


In [ ]:
def knn_predict_one(
    X_train: np.ndarray,
    y_train: np.ndarray,
    x_query: np.ndarray,
    k: int,
) -> tuple[int, float]:
    """Predict the class of a single query sample.

    Parameters
    ----------
    X_train : ndarray of shape (m, n)
    y_train : ndarray of shape (m,)
    x_query : ndarray of shape (n,)
    k : int

    Returns
    -------
    prediction : int
        0 or 1
    prob_1 : float
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_B6(knn_predict_one)

## B7 — Predict a Complete Dataset (5 points)

`knn_predict_one` predicts only one sample.

Complete `knn_predict`.

Return:
```python
predictions, probabilities  # both shape (n_test,)
```

**Hint:** Call `knn_predict_one` repeatedly.


In [ ]:
def knn_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    k: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Predict classes and class-1 probabilities for a full test set.

    Parameters
    ----------
    X_train : ndarray of shape (m, n)
    y_train : ndarray of shape (m,)
    X_test : ndarray of shape (n_test, n)
    k : int

    Returns
    -------
    predictions : ndarray of shape (n_test,)
    probabilities : ndarray of shape (n_test,)
    """
    predictions = []
    probabilities = []

    # YOUR CODE HERE

    return np.array(predictions), np.array(probabilities)


In [ ]:
# DO NOT EDIT
test_B7(knn_predict)

In [ ]:
# DO NOT EDIT — Run your KNN implementation on the exam data
y_pred_knn, y_prob_knn = knn_predict(
    X_train_scaled,
    y_train,
    X_test_scaled,
    k=7
)

print("Predictions shape:", y_pred_knn.shape)
print("Probabilities shape:", y_prob_knn.shape)

### Stage 4 — Evaluate the Predictions

Your KNN implementation can now produce predictions for the complete scaled test set.

We will next summarize those predictions with a binary confusion matrix. The matrix itself will also be implemented from scratch using NumPy Boolean masks.

## B8 — Binary Confusion Matrix from Scratch (6 points)

Do **not** use `sklearn.metrics.confusion_matrix`.

Compute:
- TN
- FP
- FN
- TP

Return exactly:

```text
[[TN, FP],
 [FN, TP]]
```

**Hint:** Boolean conditions can be combined using `&`.


In [ ]:
def confusion_matrix_binary(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> np.ndarray:
    """Build a 2x2 confusion matrix for binary labels.

    Parameters
    ----------
    y_true : ndarray of shape (m,)
    y_pred : ndarray of shape (m,)

    Returns
    -------
    cm : ndarray of shape (2, 2)
        [[TN, FP],
         [FN, TP]]
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_B8(confusion_matrix_binary)

In [ ]:
# DO NOT EDIT
print(confusion_matrix_binary(y_test, y_pred_knn))

### Stage 5 — Separate Probability Estimation from the Decision Threshold

Your KNN code already estimates `prob_1` from the labels of the nearest neighbors. Until now, `knn_predict_one` converted that probability to a class using a fixed threshold of `0.5`.

We will now make that final decision step reusable:
`KNN probability → custom threshold → predicted class`

The provided cell will apply several thresholds and reuse your confusion-matrix function. No plotting code is required.

## B9 — Custom Classification Threshold (3 points)

Complete `apply_threshold`.

It must convert class-1 probabilities into integer labels using any supplied threshold.

Return:
```python
labels  # integer ndarray, same length as probabilities
```

Then the provided cell will display confusion matrices for thresholds:
`0.3`, `0.5`, and `0.7`.


In [ ]:
def apply_threshold(
    probabilities: np.ndarray,
    threshold: float,
) -> np.ndarray:
    """Convert class-1 probabilities to hard labels with a threshold.

    Parameters
    ----------
    probabilities : ndarray of shape (m,)
    threshold : float

    Returns
    -------
    labels : ndarray of shape (m,)
        Integer labels 0 or 1.
    """
    # YOUR CODE HERE
    pass


In [ ]:
# DO NOT EDIT
test_B9(apply_threshold)

In [ ]:
# DO NOT EDIT
for threshold in [0.3, 0.5, 0.7]:
    pred_t = apply_threshold(y_prob_knn, threshold)
    print(f"Threshold = {threshold}")
    print(confusion_matrix_binary(y_test, pred_t))
    print()

### Stage 6 — Select k with Cross-Validation

The from-scratch KNN model above was demonstrated with a fixed value of `k`.

In the final stage, we will compare multiple candidate values of `k` using scikit-learn's `KNeighborsClassifier` and K-Fold Cross-Validation.

Because KNN is distance-based, scaling must be performed **inside each Cross-Validation fold**. For that reason, each candidate model will use this Pipeline:
`StandardScaler → KNeighborsClassifier`

The plotting utility will visualize the mean CV accuracy for the candidate `k` values.

## B10 — Find the Best k with scikit-learn Cross-Validation (4 points)

Complete `select_best_k`.

For every candidate `k`:
- build a Pipeline containing `StandardScaler` and `KNeighborsClassifier`,
- run K-Fold Cross-Validation,
- use accuracy,
- store the mean CV accuracy.

Return:
```python
best_k, mean_scores
```

Requirements:
- `KFold(n_splits=n_splits, shuffle=True, random_state=42)`
- `scoring="accuracy"`
- choose the `k` with the largest mean CV score.


In [ ]:
def select_best_k(
    X: np.ndarray,
    y: np.ndarray,
    k_values: np.ndarray,
    n_splits: int = 5,
) -> tuple[int, np.ndarray]:
    """Select the KNN k with the best mean CV accuracy.

    Parameters
    ----------
    X : ndarray of shape (m, n)
    y : ndarray of shape (m,)
    k_values : ndarray of candidate k values
    n_splits : int

    Returns
    -------
    best_k : int
    mean_scores : ndarray of shape (len(k_values),)
    """
    mean_scores = []

    # YOUR CODE HERE

    return best_k, np.array(mean_scores)


In [ ]:
# DO NOT EDIT
test_B10(select_best_k)

In [ ]:
# DO NOT EDIT — Run k selection on the raw dataset
# Scaling is handled inside the Pipeline within each CV fold.
X_all = cls_df.drop(columns=["churn"]).to_numpy(dtype=float)
y_all = cls_df["churn"].to_numpy()

k_values = np.arange(1, 21)
best_k, k_scores = select_best_k(X_all, y_all, k_values, n_splits=5)

print("Best k:", best_k)

plot_model_selection_scores(
    k_values,
    k_scores,
    xlabel="k",
    ylabel="Mean CV Accuracy",
    title="KNN Model Selection"
)

---
# End of Exam

Before submission:
1. Restart the kernel.
2. Run all cells from top to bottom.
3. Confirm that every completed task test prints `All tests passed!`.
4. Save the notebook using the filename requested by your instructor.

## Final Workflow Reminder

A correct machine learning solution is not only a model.
It is the complete workflow:

Data preparation → preprocessing → training → evaluation → model selection.

# Final Regression Workflow Check

Your regression pipeline should now follow:

Dataset
↓
Train/Test Split
↓
(Optional Raw Training)
↓
Fit Scaler on Train Only
↓
Transform Train/Test
↓
Train Model
↓
Evaluate on Test Data